In [1]:
import sys
print(sys.version)

3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]


In [2]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
import tensorflow as tf
from tensorflow import keras

print("TF version:", tf.__version__)
print("Keras version:", keras.__version__)

TF version: 2.10.0
Keras version: 2.10.0


In [4]:
import cv2
from keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np
import tensorflow as tf
import keras

In [5]:
model = keras.models.load_model("asl_classifier.h5")

In [6]:
labels_dict = {0:'0', 
                 1:'A', 
                 2:'B', 
                 3:'C', 
                 4:'D', 
                 5:'E',
                 6:'F',
                 7:'G',
                 8:'H',
                 9:'I',
                 10:'J',
                 11:'K',
                 12:'L',
                 13:'M',
                 14:'N',
                 15:'O',
                 16:'P',
                 17:"Q",
                 18:'R',
                 19:'S',
                 20:'T', 
                 21:'U', 
                 22:'V',
                 23:'W',
                 24:'X',
                 25:'Y',
                 26:'Z'}
color_dict=(0,255,0)
x=0
y=0
w=64
h=64

# Fully Real-Time

In [ ]:
img_size=128
minValue = 70
source=cv2.VideoCapture(0)
count = 0
string = " "
prev = " "
prev_val = 0
while(True):
    ret,img=source.read()
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    #cv2.rectangle(img,(x,y),(x+w,y+h),color_dict,2)
    cv2.rectangle(img,(24,24),(250 , 250),color_dict,2)
    crop_img=gray[24:250,24:250]
    count = count + 1
    if(count % 100 == 0):
        prev_val = count
    cv2.putText(img, str(prev_val//100), (300, 150),cv2.FONT_HERSHEY_SIMPLEX,1.5,(255,255,255),2) 
    blur = cv2.GaussianBlur(crop_img,(5,5),2)
    th3 = cv2.adaptiveThreshold(blur,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,11,2)
    ret, res = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    resized=cv2.resize(res,(img_size,img_size))
    normalized=resized/255.0
    reshaped=np.reshape(normalized,(1,img_size,img_size,1))
    result = model.predict(reshaped)
    #print(result)
    label=np.argmax(result,axis=1)[0]
    if(count == 300):
        count = 99
        prev= labels_dict[label] 
        if(label == 0):
               string = string + " "
            #if(len(string)==1 or string[len(string)] != " "):
             
        else:
                string = string + prev
    
    cv2.putText(img, prev, (24, 14),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2) 
    cv2.putText(img, string, (275, 50),cv2.FONT_HERSHEY_SIMPLEX,0.8,(200,200,200),2)
    cv2.imshow("Gray",res)    
    cv2.imshow('LIVE',img)
    key=cv2.waitKey(1)
    
 
    if(key==27):#press Esc. to exit
        break
print(string)        
cv2.destroyAllWindows()
source.release()

cv2.destroyAllWindows()

1/1 [==============================] - 0s 43ms/step


In [ ]:
# pip install gTTS

In [ ]:
from gtts import gTTS 
import os 
from playsound import playsound

# 1. Create a dictionary mapping English words to Urdu equivalents
# You can expand this dictionary with as many words as you need your system to recognize.
english_to_urdu_dict = {
    "hello": "ہیلو",
    "how": "کیسے",
    "are": "ہیں",
    "you": "آپ",
    "hi": "سلام",
    "good": "اچھا",
    "morning": "صبح بخیر",
    "bye": "خدا حافظ",
    "thanks": "شکریہ",
    "yes": "ہاں",
    "no": "نہیں"
}

# The 'string' variable contains the predicted English letters combined from the ASL hand gestures.
# Example: If you signed H-E-L-L-O, string = "HELLO"
# Let's clean up white spaces and make it lowercase to match our dictionary keys.
predicted_english_text = string.strip().lower()

# 2. Convert predicted English letters/words into Urdu words
urdu_words = []
for word in predicted_english_text.split():
    # If the word exists in our dictionary, append the Urdu version. 
    # Otherwise, just append the original English word to prevent losing data.
    translated_word = english_to_urdu_dict.get(word, word)
    urdu_words.append(translated_word)

# Join the translated words back into a single sentence
urdu_text = " ".join(urdu_words)

# Print it out just so you can see the translation in your notebook cell
print(f"Original English: {predicted_english_text}")
print(f"Translated to Urdu: {urdu_text}")

# 3. Use gTTS with Urdu language (lang='ur')
if urdu_text.strip() != "":
    language = 'ur'
    
    # Passing the text and language to the engine
    myobj = gTTS(text=urdu_text, lang=language, slow=False) 
      
    # Saving the converted audio in a mp3 file
    audio_file = "urdu_welcome.mp3"
    
    # Remove old file if it exists to prevent permission errors when saving
    if os.path.exists(audio_file):
        os.remove(audio_file)
        
    myobj.save(audio_file) 
      
    # 4. Generate Urdu speech output by playing the converted file
    try:
        # playsound is generally more reliable across different systems
        playsound(audio_file)
    except Exception as e:
        # Fallback to the system's default media player
        print(f"Playsound failed, relying on system fallback: {e}")
        os.system(audio_file)
else:
    print("No valid ASL letters were detected to translate!")
